# Section 06 lab — Set operations, pivot and data quality

Run the section setup once. Work through each activity from Step 1 onward; every step explains why it is needed before the code runs.


## Section setup — run once

Create one local Spark session and reuse it throughout this section.


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
print('Spark:', spark.version, '| master:', spark.sparkContext.master)


## Exercise 1 — Append rows with union and unionByName

Combine DataFrames only after confirming that their columns are compatible.


### Step 1 — Union DataFrames with the same column order

union matches columns by position.


In [ ]:
january = spark.createDataFrame([('O1', 'Jan', 500.0), ('O2', 'Jan', 750.0)], ['order_id', 'month', 'revenue'])
february = spark.createDataFrame([('O3', 'Feb', 1200.0)], ['order_id', 'month', 'revenue'])
combined = january.union(february)
combined.show()


### Step 2 — Create a DataFrame with a different column order

The names are the same, but their positions differ.


In [ ]:
march = spark.createDataFrame([('Mar', 'O4', 900.0)], ['month', 'order_id', 'revenue'])
march.show()


### Step 3 — Match by name

unionByName places values under the correct columns.


In [ ]:
all_months = combined.unionByName(march)
all_months.show()
assert all_months.count() == 4


## Exercise 2 — Compare row sets

Identify common and missing rows after compatible schemas have been established.


### Step 1 — Create two row sets

B appears twice in the first set so duplicate behavior is visible.


In [ ]:
first = spark.createDataFrame([('A',), ('B',), ('B',), ('C',)], ['code'])
second = spark.createDataFrame([('B',), ('C',), ('D',)], ['code'])


### Step 2 — Find common rows

intersect returns values present in both sets.


In [ ]:
common = first.intersect(second)
common.orderBy("code").show()


### Step 3 — Subtract rows with duplicate awareness

exceptAll removes matching occurrences and keeps the extra B.


In [ ]:
only_first = first.exceptAll(second)
only_first.orderBy("code").show()
assert [row['code'] for row in only_first.orderBy('code').collect()] == ['A', 'B']


## Exercise 3 — Pivot an aggregation

Turn known category values into columns after grouping has already been understood.


### Step 1 — Create city and category sales

Each city has sales in more than one category.


In [ ]:
sales = spark.createDataFrame([('Pune', 'Accessories', 1000.0), ('Pune', 'Audio', 1200.0), ('Mumbai', 'Accessories', 750.0), ('Mumbai', 'Electronics', 2400.0)], ['city', 'category', 'revenue'])


### Step 2 — Pivot category values into columns

The explicit value list keeps the output columns predictable.


In [ ]:
from pyspark.sql import functions as F
matrix = sales.groupBy('city').pivot('category', ['Accessories', 'Audio', 'Electronics']).agg(F.sum('revenue'))


### Step 3 — Replace missing combinations with zero

A missing city-category combination is displayed clearly.


In [ ]:
matrix.fillna(0).orderBy("city").show()
assert matrix.count() == 2


## Exercise 4 — Classify data-quality failures

Apply typed rules only after transformation, aggregation and join behavior are familiar.


### Step 1 — Create valid and invalid source rows

The errors cover missing, malformed and non-positive values.


In [ ]:
raw = spark.createDataFrame([('T1', '2026-10-01', '2', '500.00'), ('T2', 'bad-date', '1', '750.00'), ('T3', '2026-10-02', 'two', '100.00'), ('T4', '2026-10-03', '-1', '250.00')], ['transaction_id', 'transaction_date', 'quantity', 'unit_price'])
raw.show()


### Step 2 — Convert fields safely

try_cast turns malformed values into null so the job can continue.


In [ ]:
from pyspark.sql import functions as F
typed = (raw.withColumn('order_date', F.expr('try_cast(transaction_date as date)')).withColumn('qty', F.expr('try_cast(quantity as int)')).withColumn('price', F.expr('try_cast(unit_price as decimal(12,2))')))
typed.show()


### Step 3 — Attach rejection reasons

Every failed rule remains explainable.


In [ ]:
reason = F.concat_ws('; ', F.when(F.col('order_date').isNull(), 'invalid date'), F.when(F.col('qty').isNull(), 'invalid quantity'), F.when(F.col('qty') <= 0, 'non-positive quantity'), F.when(F.col('price').isNull() | (F.col('price') < 0), 'invalid price'))
classified = typed.withColumn('reject_reason', reason)
valid = classified.filter(F.col('reject_reason') == '')
rejected = classified.filter(F.col('reject_reason') != '')
rejected.select('transaction_id', 'reject_reason').show(truncate=False)
assert valid.count() == 1 and rejected.count() == 3


## Exercise 5 — Inspect and improve an execution plan

Use operations already covered—join and groupBy—before examining broadcast, exchange and cache behavior.


### Step 1 — Build a familiar join and aggregation

A small product table enriches transaction rows before category totals.


In [ ]:
from pyspark.sql import functions as F
transactions = spark.createDataFrame([(1, 'P1', 500.0), (2, 'P2', 1200.0), (3, 'P1', 750.0)], ['transaction_id', 'product_id', 'revenue'])
products = spark.createDataFrame([('P1', 'Accessories'), ('P2', 'Audio')], ['product_id', 'category'])
summary = transactions.join(F.broadcast(products), 'product_id').groupBy('category').agg(F.sum('revenue').alias('revenue'))


### Step 2 — Read the formatted plan

Look for BroadcastHashJoin and Exchange.


In [ ]:
summary.explain('formatted')
summary.show()


### Step 3 — Cache only reused results

Materialize once, reuse twice, then release the memory.


In [ ]:
reused = summary.cache()
reused.count()
reused.orderBy(F.desc("revenue")).show()
reused.agg(F.sum("revenue")).show()
reused.unpersist()
assert reused.is_cached is False


## Exercise 6 — Rank rows within groups using window functions

Keep every employee row while calculating a position inside each department.


### Step 1 — Create employees with tied salaries

Equal salaries make the difference between row_number, rank and dense_rank visible.


In [ ]:
employees = spark.createDataFrame([('Engineering', 'Asha', 95000), ('Engineering', 'Bilal', 90000), ('Engineering', 'Chen', 90000), ('Sales', 'Divya', 80000), ('Sales', 'Eshan', 75000)], ['department', 'employee_name', 'salary'])
employees.show()


### Step 2 — Define the department ranking window

partitionBy restarts the ranking for each department; orderBy places higher salaries first.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
ranking_window = Window.partitionBy('department').orderBy(F.desc('salary'))


### Step 3 — Compare three ranking functions

row_number is always unique, rank leaves gaps after ties, and dense_rank does not leave gaps.


In [ ]:
ranked = (employees.withColumn('row_number', F.row_number().over(ranking_window)).withColumn('rank', F.rank().over(ranking_window)).withColumn('dense_rank', F.dense_rank().over(ranking_window)))
ranked.orderBy('department', 'row_number').show()


## Exercise 7 — Calculate analytical window measures

Add department totals, averages, running totals and previous-row values without collapsing detail rows.


### Step 1 — Create dated department sales

Several rows per department allow partition and running calculations.


In [ ]:
sales = spark.createDataFrame([('2026-10-01', 'A', 500.0), ('2026-10-02', 'A', 750.0), ('2026-10-03', 'A', 250.0), ('2026-10-01', 'B', 1200.0), ('2026-10-02', 'B', 800.0)], ['sale_date', 'department', 'revenue'])


### Step 2 — Define partition and running windows

The partition window sees the full department; the running frame ends at the current row.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
department_window = Window.partitionBy('department')
running_window = Window.partitionBy('department').orderBy('sale_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
ordered_window = Window.partitionBy('department').orderBy('sale_date')


### Step 3 — Add analytical columns

Window aggregates preserve every source row while adding context from related rows.


In [ ]:
analysed = (sales.withColumn('department_total', F.sum('revenue').over(department_window)).withColumn('department_average', F.avg('revenue').over(department_window)).withColumn('running_revenue', F.sum('revenue').over(running_window)).withColumn('previous_revenue', F.lag('revenue').over(ordered_window)))
analysed.orderBy('department', 'sale_date').show()
